## Deep Learning, DL Models, and Applications

```python
# Task 7: ResNet-50 Style Residual Bottleneck Block and Grouped
# Convolutions
# ● Objective: Master deep architectural scaling paradigms by constructing bottleneck
# projection networks and multi-channel grouped/depthwise separable filters.
# ● Required Tech Stack: PyTorch, TorchVision, PyTorch Profiler.
# ● Task Description: Students must build a custom ResNet-50 bottleneck block featuring
# depthwise separable convolutions and learnable skip projection layers. They will run
# PyTorch Profiler to analyze and optimize memory footprints, parameter counts, and
# floating-point execution steps (FLOPs) under different channel scaling factors.
```

In [154]:
#Step 1: Import Libraries
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity

torch.manual_seed(42)


In [155]:
#Step 2: Depthwise Separable Convolution
class DepthwiseSeparableConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            padding=1,
            groups=in_channels
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        x = self.depthwise(x)

        x = self.pointwise(x)

        return x

In [156]:
#Step 3: Bottleneck Block
class BottleneckBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        # 1x1 Reduce
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

        # Depthwise Separable Conv
        self.conv2 = DepthwiseSeparableConv(
            out_channels,
            out_channels
        )

        # 1x1 Expand
        self.conv3 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=1
        )

        # Skip Projection
        self.skip = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        identity = self.skip(x)

        out = self.relu(self.conv1(x))

        out = self.relu(self.conv2(out))

        out = self.conv3(out)

        out = out + identity

        out = self.relu(out)

        return out

In [158]:
#Step 4: Create Input
X = torch.randn(1,64,32,32)

In [159]:
#Step 5: Test
model = BottleneckBlock(64,64)

output = model(X)

print(output.shape)

torch.Size([1, 64, 32, 32])


In [160]:
#Step 6: Count Parameters
total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)

Total Parameters: 17280


In [161]:
#Step 7: PyTorch Profiler
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True
) as prof:

    output = model(X)

print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=10
))

------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                          Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  aten::conv2d         0.39%      33.553us        74.40%       6.327ms       1.265ms             5  
             aten::convolution         1.22%     103.582us        74.01%       6.294ms       1.259ms             5  
            aten::_convolution         2.28%     193.967us        72.79%       6.190ms       1.238ms             5  
             aten::thnn_conv2d         0.22%      18.289us        65.13%       5.538ms       1.385ms             4  
    aten::_slow_conv2d_forward        61.94%       5.267ms        64.91%       5.520ms       1.380ms             4  
                    aten::relu        23.17%       1.970ms      

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [162]:
channels = [32, 64, 128]

for c in channels:

    model = BottleneckBlock(c, c)

    x = torch.randn(1, c, 32, 32)

    out = model(x)

    params = sum(p.numel() for p in model.parameters())

    print(f"Channels: {c}")

    print(f"Output Shape: {out.shape}")

    print(f"Parameters: {params}")

    print("-" * 30)

Channels: 32
Output Shape: torch.Size([1, 32, 32, 32])
Parameters: 4544
------------------------------
Channels: 64
Output Shape: torch.Size([1, 64, 32, 32])
Parameters: 17280
------------------------------
Channels: 128
Output Shape: torch.Size([1, 128, 32, 32])
Parameters: 67328
------------------------------
